In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine

load_dotenv(find_dotenv(usecwd=True))
engine = create_engine(os.environ['DATABASE_URL'])

In [ ]:
pasta_csv = 'Arquivos CSV/'

dim_agente_csv = pd.read_csv(pasta_csv + 'dim_agente.csv')[['nome_agente']]
dim_filial_csv = pd.read_csv(pasta_csv + 'dim_filial.csv')[['loja', 'sigla_uf']]
dim_carac_csv  = pd.read_csv(pasta_csv + 'dim_caracteristicas.csv')[['lista_id', 'qualificacao', 'comportamento']]

In [ ]:
assert dim_agente_csv['nome_agente'].is_unique, 'nome_agente repetido no CSV'
assert not dim_filial_csv.duplicated().any(), 'linha repetida em dim_filial'
assert not dim_carac_csv.duplicated().any(), 'linha repetida em dim_caracteristicas'

for nome, df in [('dim_agente', dim_agente_csv), ('dim_filial', dim_filial_csv), ('dim_caracteristicas', dim_carac_csv)]:
    assert df.notna().all().all(), f'{nome} tem valores nulos'

assert dim_agente_csv['nome_agente'].str.startswith('Operador').all(), 'há nome de agente que não é mascarado'
assert dim_filial_csv['loja'].str.startswith('Filial').all(), 'há loja que não é mascarada'

for tabela in ['dim_agente', 'dim_filial', 'dim_caracteristicas']:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {tabela}', engine)['n'][0]
    assert n == 0, f'{tabela} já tem {n} linhas: esta carga já foi feita?'

In [ ]:
with engine.begin() as conexao:
    dim_agente_csv.to_sql('dim_agente', conexao, if_exists='append', index=False)
    dim_filial_csv.to_sql('dim_filial', conexao, if_exists='append', index=False)
    dim_carac_csv.to_sql('dim_caracteristicas', conexao, if_exists='append', index=False)

In [ ]:
for tabela in ['dim_agente', 'dim_filial', 'dim_caracteristicas', 'dim_tempo']:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {tabela}', engine)['n'][0]
    print(tabela, n)

In [ ]:
pd.read_sql('SELECT * FROM dim_agente ORDER BY id_agente LIMIT 5', engine)

In [ ]:
pd.read_sql('SELECT * FROM dim_filial ORDER BY id_filial LIMIT 5', engine)